# M5: QAOA completo en N pequeño (Camino B)

Parámetros según el plan:
- N=3 (9 qubits), N=4 (16 qubits)
- `reps ∈ {1,2,3}`, COBYLA `maxiter` 100-200, `shots=4096`
- Inicialización pequeña y fija por semilla, semillas 0-4
- `λ = 2	imesλ^*` (del estudio M4)
- Métricas en `Result.extras`: mejor costo factible, fracción factible, probabilidad óptimo, energía penalizada final, historial, tiempo, qubits/profundidad/puertas de 2 qubits

**Resultado:** siempre comparamos con la fuerza bruta (24/32). Aunque la fracción factible sea pequeña, si se encuentra el óptimo hay que reportarlo con honestidad.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../results/qaoa_m5.csv')
df

In [ ]:
df.groupby(['n','reps'])[['best_feasible_cost','feasible_fraction','optimal_prob','seconds']].mean()

## Resumen por (N, reps)

In [ ]:
for n in [3,4]:
    print(f'=== N={n} (optimo {24 if n==3 else 32}) ===')
    for reps in [1,2,3]:
        sub = df[(df.n==n) & (df.reps==reps)]
        found_opt = (sub.best_feasible_cost == (24 if n==3 else 32)).sum()
        print(f'  reps={reps}: mejor_costo_medio={sub.best_feasible_cost.mean():.1f}, ff_medio={sub.feasible_fraction.mean():.4f}, encuentra_optimo={found_opt}/5, tiempo_medio={sub.seconds.mean():.1f}s')
        print(f'          qubits={sub.qubits.iloc[0]}, depth={sub.depth.iloc[0]}, 2q={sub.two_qubit_gates.iloc[0]}')

## Gráficas para las diapositivas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8,3))

# N=3
sub3 = df[df.n==3].groupby('reps').mean(numeric_only=True)
axes[0].bar(sub3.index.astype(str), sub3.best_feasible_cost)
axes[0].axhline(24, ls='--', color='r', label='óptimo 24')
axes[0].set_title('N=3 - mejor costo factible')
axes[0].set_xlabel('reps')
axes[0].legend()

# N=4
sub4 = df[df.n==4].groupby('reps').mean(numeric_only=True)
axes[1].bar(sub4.index.astype(str), sub4.best_feasible_cost)
axes[1].axhline(32, ls='--', color='r', label='óptimo 32')
axes[1].set_title('N=4 - mejor costo factible')
axes[1].set_xlabel('reps')
axes[1].legend()

plt.tight_layout()
plt.savefig('../docs/figures/qaoa_m5_cost.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5,3))
for n in [3,4]:
    sub = df[df.n==n].groupby('reps').mean(numeric_only=True)
    ax.plot(sub.index, sub.feasible_fraction, marker='o', label=f'N={n}')
ax.set_xlabel('reps')
ax.set_ylabel('fracción factible')
ax.set_title('QAOA - fracción factible (shots=4096)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.savefig('../docs/figures/qaoa_m5_feasible_fraction.png', dpi=150, bbox_inches='tight')
plt.show()